In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, mean_absolute_error, r2_score, mean_squared_error
import re

In [2]:
#Import dei dataset
df_melbourne = pd.read_csv('Melbourne_housing.csv', low_memory=False)

df_california = pd.read_csv("california_housing_data.csv")

df_cars = pd.read_csv("Cars Datasets 2025.csv", encoding='latin-1')


**PULIZIA DATI MELBOURNE HOUSING**

In [3]:
columns_to_use = ['Suburb', 'Rooms', 'Type', 'Method', 'SellerG', 'Regionname', 'Propertycount', 'Distance', 'Bedroom', 'Bathroom', 'Postcode','Car', 'Price']
df_melbourne = df_melbourne[columns_to_use]

columns_to_fill_0 = ['Car', 'Bathroom', 'Bedroom']
df_melbourne[columns_to_fill_0] = df_melbourne[columns_to_fill_0].fillna(0)

df_melbourne = df_melbourne.replace([np.inf, -np.inf], np.nan).dropna()

df_melbourne = pd.get_dummies(df_melbourne, drop_first=True)

**PULIZIA DATI CARS**

In [4]:
df_cars = df_cars.drop(columns=['Engines', 'Seats', 'Fuel Types'])

#normalizzazione del brand name
def normalize_brand(name):
    name = str(name).strip().lower()
    name = re.sub(r"\s+", " ", name)

    if name in ["bmw", "gmc"]:
        return name.upper()

    elif "rolls" in name:
        return "Rolls-Royce"

    elif "mercedes" in name:
        return "Mercedes-Benz"

    return name.title()

df_cars["Company Names"] = df_cars["Company Names"].apply(normalize_brand)

df_cars.groupby("Company Names").count()

#conversione valori stringa in numeri
def extract_numeric(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    m = re.findall(r"[-+]?\d*\.?\d+", s)
    if not m:
        return np.nan
    return float(m[0])

numeric_cols_raw = [
    "CC/Battery Capacity",
    "HorsePower",
    "Total Speed",
    "Performance(0 - 100 )KM/H",
    "Cars Prices",
    "Torque"
]

for col in numeric_cols_raw:
    if col in df_cars.columns:
        df_cars[col] = df_cars[col].apply(extract_numeric)

#elenco colonna numeriche
numeric_cols = df_cars.select_dtypes(include=["float", "int"]).columns.tolist()

#rimozione e sostituzione colonne valori nulli essendo di poco conto
df_cars[numeric_cols].fillna(df_cars[numeric_cols].mean())
df_cars[numeric_cols].dropna()

df_cars = pd.get_dummies(df_cars, drop_first=True)

**OUTLIERS**

In [5]:
#rimozione outlier
numeric_cols = df_melbourne.select_dtypes(include=["float", "int"]).columns.tolist()

for col in numeric_cols:
    Q1 = df_melbourne[col].quantile(0.2)
    Q3 = df_melbourne[col].quantile(0.8)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    df_melbourne[col] = np.where(df_melbourne[col].between(lower, upper), df_melbourne[col], np.nan)

df_melbourne = df_melbourne.dropna()

In [6]:
#rimozione outlier
numeric_cols = df_california.select_dtypes(include=["float", "int"]).columns.tolist()

for col in numeric_cols:
    Q1 = df_california[col].quantile(0.2)
    Q3 = df_california[col].quantile(0.8)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    df_california[col] = np.where(df_california[col].between(lower, upper), df_california[col], np.nan)

df_california = df_california.dropna()

In [7]:
#rimozione outlier
numeric_cols = df_cars.select_dtypes(include=["float", "int"]).columns.tolist()

for col in numeric_cols:
    Q1 = df_cars[col].quantile(0.2)
    Q3 = df_cars[col].quantile(0.8)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    df_cars[col] = np.where(df_cars[col].between(lower, upper), df_cars[col], np.nan)

df_cars = df_cars.dropna()

**DISTRIBUZIONE IPERPARAMETRI PER GRID SEARCH**

In [8]:
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],   
    'max_depth': [3, 5],             
    'subsample': [0.8, 1.0]          
}

**MELBOURNE HOUSING ML**

In [9]:
X = df_melbourne.drop('Price', axis='columns')
y= df_melbourne["Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    eval_metric='logloss',    
    use_label_encoder=False,
    random_state=42,
    n_jobs=3
)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,         
    scoring='neg_mean_squared_error',   
    verbose=1,            
    n_jobs=3            
)

print("\n--- Inizio Tuning (Attendere...) ---")
grid_search.fit(X_train, y_train)

print("\n--- Tuning Completato ---")
print(f"Migliori Parametri Trovati: {grid_search.best_params_}")
best_rmse = np.sqrt(-grid_search.best_score_)
print(f"Miglior RMSE (Media CV): {best_rmse:.4f}")

best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test)

print("\n--- Report Finale (Test Set) ---")
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE Test Set: {rmse_test:.4f}")

r2 = r2_score(y_test, y_pred)
print(f"R2 Test Set: {r2:.4f}")


--- Inizio Tuning (Attendere...) ---
Fitting 5 folds for each of 16 candidates, totalling 80 fits


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [17:36:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [17:36:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [17:36:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [17:36:47] WARN


--- Tuning Completato ---
Migliori Parametri Trovati: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
Miglior RMSE (Media CV): 225279.6892

--- Report Finale (Test Set) ---
RMSE Test Set: 225384.6189
R2 Test Set: 0.7760


**CALIFORNIA HOUSING ML**

In [10]:
X = df_california.drop(columns=["MedHouseVal"])
y= df_california["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    eval_metric='logloss',    
    use_label_encoder=False,
    random_state=42,
    n_jobs=3
)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,         
    scoring='neg_mean_squared_error',   
    verbose=1,            
    n_jobs=3             
)

print("\n--- Inizio Tuning (Attendere...) ---")
grid_search.fit(X_train, y_train)

print("\n--- Tuning Completato ---")
print(f"Migliori Parametri Trovati: {grid_search.best_params_}")
best_rmse = np.sqrt(-grid_search.best_score_)
print(f"Miglior RMSE (Media CV): {best_rmse:.4f}")

best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test)

print("\n--- Report Finale (Test Set) ---")
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE Test Set: {rmse_test:.4f}")

r2 = r2_score(y_test, y_pred)
print(f"R2 Test Set: {r2:.4f}")


--- Inizio Tuning (Attendere...) ---
Fitting 5 folds for each of 16 candidates, totalling 80 fits


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [17:37:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [17:37:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [17:37:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [17:37:17] WARN


--- Tuning Completato ---
Migliori Parametri Trovati: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
Miglior RMSE (Media CV): 0.4527

--- Report Finale (Test Set) ---
RMSE Test Set: 0.4614
R2 Test Set: 0.8238


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [17:37:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


**CARS PARAM ML**

In [11]:
X = df_cars.drop('Cars Prices', axis='columns')
y= df_cars["Cars Prices"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    eval_metric='logloss',    
    use_label_encoder=False,
    random_state=42,
    n_jobs=3
)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,         
    scoring='neg_mean_squared_error',   
    verbose=1,            
    n_jobs=3            
)

print("\n--- Inizio Tuning (Attendere...) ---")
grid_search.fit(X_train, y_train)

print("\n--- Tuning Completato ---")
print(f"Migliori Parametri Trovati: {grid_search.best_params_}")
best_rmse = np.sqrt(-grid_search.best_score_)
print(f"Miglior RMSE (Media CV): {best_rmse:.4f}")

best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test)

print("\n--- Report Finale (Test Set) ---")
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE Test Set: {rmse_test:.4f}")

r2 = r2_score(y_test, y_pred)
print(f"R2 Test Set: {r2:.4f}")


--- Inizio Tuning (Attendere...) ---
Fitting 5 folds for each of 16 candidates, totalling 80 fits


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [17:37:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [17:37:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [17:37:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [17:37:22] WARN


--- Tuning Completato ---
Migliori Parametri Trovati: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
Miglior RMSE (Media CV): 14.0068

--- Report Finale (Test Set) ---
RMSE Test Set: 15.0454
R2 Test Set: 0.7623
